In [ ]:
###
# 1. 環境のセットアップ
###

import sys
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

ROOT_PATH = Path('/content/drive/MyDrive/cnn-hands-on')
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

os.chdir(ROOT_PATH)

# 日本語フォント対応
!pip install -q japanize-matplotlib
import japanize_matplotlib

print(f"✅ 環境セットアップ完了！現在のディレクトリ: {Path.cwd()}")

# 1. 全結合層とは

## CNNの構造おさらい

1. **畳み込み層**: 特徴を抽出
2. **活性化関数**: 非線形性を追加
3. **プーリング層**: サイズを縮小
4. **全結合層**: 分類を実行 ← **今回**

全結合層は抽出した特徴を使って「何であるか」を判断する

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 全結合層の概念を可視化
fig, ax = plt.subplots(figsize=(10, 6))

# 入力層（4ノード）
input_nodes = 4
output_nodes = 2

input_y = np.linspace(0.2, 0.8, input_nodes)
output_y = np.linspace(0.35, 0.65, output_nodes)

# 全結合: すべての入力とすべての出力が接続
for iy in input_y:
    for oy in output_y:
        ax.plot([0.2, 0.8], [iy, oy], 'b-', alpha=0.3, linewidth=1)

# ノードを描画
for y in input_y:
    ax.plot(0.2, y, 'o', markersize=20, color='lightblue', markeredgecolor='black')
for y in output_y:
    ax.plot(0.8, y, 'o', markersize=20, color='lightgreen', markeredgecolor='black')

ax.text(0.2, 0.05, f'入力\n({input_nodes}次元)', ha='center', fontsize=12)
ax.text(0.8, 0.05, f'出力\n({output_nodes}次元)', ha='center', fontsize=12)
ax.set_title('全結合層: すべてのニューロンが全入力と接続', fontsize=14)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
plt.show()

print(f"パラメータ数 = {input_nodes} × {output_nodes} + {output_nodes} = {input_nodes * output_nodes + output_nodes}")
print("（重みW + バイアスb）")

# 2. Flattenの実装

In [ ]:
# Flatten（平坦化）の概念
# 畳み込み・プーリング後の特徴マップは3次元 (C, H, W)
# 全結合層に入力するには1次元にする必要がある

# 例: (16, 7, 7) → (784,)
example_shape = (16, 7, 7)
flattened_size = 16 * 7 * 7
print(f"特徴マップ: {example_shape}")
print(f"Flatten後: ({flattened_size},)")

In [ ]:
###
# 演習1: Flattenを手動で実装
###

def flatten(x):
    """
    3次元配列を1次元に変換
    入力: (C, H, W)
    出力: (C * H * W,)
    """
    return x.?(?)

In [ ]:
# テスト
x = np.random.randn(16, 7, 7)
flat = flatten(x)
print(f"入力: {x.shape}")
print(f"Flatten後: {flat.shape}")
print(f"期待される出力: (784,)")

---

# 3. 全結合層の実装

## 全結合層の数式

$y = Wx + b$

- $x$: 入力ベクトル (N次元)
- $W$: 重み行列 (M × N)
- $b$: バイアスベクトル (M次元)
- $y$: 出力ベクトル (M次元)

In [ ]:
###
# 演習2: 全結合層を手動で実装
###

def linear(x, W, b):
    """
    全結合層の順伝播
    x: 入力 (N,)
    W: 重み (M, N)
    b: バイアス (M,)
    出力: (M,)
    """
    return np.?(?, ?) + ?

In [ ]:
# テスト
x = np.random.randn(784)
W = np.random.randn(10, 784)
b = np.random.randn(10)
out = linear(x, W, b)

print(f"入力: {x.shape}")
print(f"重み: {W.shape}")
print(f"バイアス: {b.shape}")
print(f"出力: {out.shape}")
print(f"期待される出力: (10,)")

---

In [ ]:
# np.dotの使い方
W = np.array([[1, 2], [3, 4]])
x = np.array([1, 1])
result = np.dot(W, x)
print(f"W = \n{W}")
print(f"x = {x}")
print(f"np.dot(W, x) = {result}")
print(f"\n計算: [1*1 + 2*1, 3*1 + 4*1] = [3, 7]")

# 4. PyTorchでの全結合層

In [ ]:
import torch
import torch.nn as nn

# nn.Flatten: バッチ次元を保持したままFlatten
flatten = nn.Flatten()

# (B, C, H, W) → (B, C*H*W)
x = torch.randn(1, 16, 7, 7)
flat = flatten(x)
print(f"入力: {x.shape}")
print(f"Flatten後: {flat.shape}")

In [ ]:
# nn.Linear: 全結合層
fc = nn.Linear(784, 10)

# パラメータの確認
print(f"重みの形状: {fc.weight.shape}")
print(f"バイアスの形状: {fc.bias.shape}")

# 推論
out = fc(flat)
print(f"出力: {out.shape}")

---

In [ ]:
###
# 発展: 複数の全結合層（隠れ層を追加）
###

# 784 → 128 → 10
fc1 = nn.Linear(784, 128)
relu = nn.ReLU()
fc2 = nn.Linear(128, 10)

x = torch.randn(1, 784)
h = relu(fc1(x))  # 隠れ層
out = fc2(h)      # 出力層

print(f"入力: {x.shape}")
print(f"隠れ層後: {h.shape}")
print(f"出力: {out.shape}")

# 5. ネットワークの構築

## これまで学んだ部品

| 層 | 役割 | PyTorch |
|---|---|---|
| 畳み込み | 特徴抽出 | `nn.Conv2d` |
| 活性化 | 非線形性 | `nn.ReLU` |
| プーリング | 縮小 | `nn.MaxPool2d` |
| 平坦化 | 1D変換 | `nn.Flatten` |
| 全結合 | 分類 | `nn.Linear` |

In [ ]:
###
# nn.Sequentialで構築
###

model = nn.Sequential(
    # 畳み込みブロック1
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    
    # 畳み込みブロック2
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    
    # 分類部分
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 10)
)

print(model)

In [ ]:
# サイズの変化を確認
x = torch.randn(1, 1, 28, 28)
print(f"入力:     {x.shape}")

# 各層を順番に適用してサイズを確認
for i, layer in enumerate(model):
    x = layer(x)
    print(f"{layer.__class__.__name__}: {x.shape}")

---

In [ ]:
###
# nn.Moduleで構築
###

import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(32 * 7 * 7, 10)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.fc(x)
        return x

model = SimpleCNN()
print(model)

In [ ]:
# 使い方
x = torch.randn(1, 1, 28, 28)
out = model(x)
print(f"入力: {x.shape}")
print(f"出力: {out.shape}")

---

In [ ]:
###
# 演習3: ネットワークを構築しよう
# 28×28のグレースケール画像を10クラスに分類するCNNを構築せよ
###

model = nn.Sequential(
    # Conv → ReLU → Pool (ブロック1)
    nn.Conv2d(?, ?, kernel_size=3, padding=1),
    nn.?(),
    nn.MaxPool2d(?),
    
    # Conv → ReLU → Pool (ブロック2)
    nn.Conv2d(?, ?, kernel_size=3, padding=1),
    nn.?(),
    nn.MaxPool2d(?),
    
    # 分類
    nn.?(),
    nn.Linear(?, ?)
)

In [ ]:
# テスト
x = torch.randn(1, 1, 28, 28)
out = model(x)
print(f"入力: {x.shape}")
print(f"出力: {out.shape}")
print(f"期待される出力: torch.Size([1, 10])")

# 6. 推論の実行

In [ ]:
# 完成したモデルを使用
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 10)
)

# ダミー入力（バッチサイズ1の28×28画像）
x = torch.randn(1, 1, 28, 28)

# 推論モード
model.eval()

# 予測
with torch.no_grad():
    output = model(x)
    
print(f"出力形状: {output.shape}")
print(f"各クラスのスコア: {output}")

# 最も高いスコアのクラスを取得
predicted = output.argmax(dim=1)
print(f"予測クラス: {predicted.item()}")

In [ ]:
# 複数の画像を同時に推論（バッチ処理）
batch_size = 4
x = torch.randn(batch_size, 1, 28, 28)

with torch.no_grad():
    output = model(x)

print(f"入力: {x.shape}")
print(f"出力: {output.shape}")

predicted = output.argmax(dim=1)
print(f"予測クラス: {predicted.tolist()}")

---

In [ ]:
# パラメータ数の確認
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"総パラメータ数: {total_params:,}")
print(f"学習可能パラメータ数: {trainable_params:,}")

# 各層のパラメータ数
print("\n各層のパラメータ数:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.numel():,}")

# 7. まとめ

- **Flatten（平坦化）**: 3次元の特徴マップ (C, H, W) を1次元に変換。学習パラメータなし
- **全結合層**: $y = Wx + b$ ですべての入力と出力を接続。分類を実行
- **nn.Sequential**: 層を順番に並べてネットワークを構築。シンプルな構造に便利
- **nn.Module**: クラスとして定義。複雑な構造も表現可能

**次回予告**: 損失関数とパラメータ更新の基礎

### 暇な人向け

In [ ]:
###
# 発展課題: 3チャンネル（RGB）画像用のネットワークを構築してみよう
# 入力: (3, 32, 32) のRGB画像
# 出力: 10クラス
###

# ヒント:
# - 最初のConv2dの入力チャンネルを3にする
# - Linearの入力サイズを再計算する
#   (32 -> 16 -> 8 とPoolingでサイズが変わる)

model_rgb = nn.Sequential(
    # ここにネットワークを構築
)

In [ ]:
# テスト
x = torch.randn(1, 3, 32, 32)
# out = model_rgb(x)
# print(f"入力: {x.shape}")
# print(f"出力: {out.shape}")